**Table of contents:**<a id='toc0_'></a>    
- [**LlamaIndex : créer son RAG en quelques lignes de code**](#toc1_)    
  - [Introduction](#toc1_1_)    
  - [1) LlamaIndex : un framework pensé pour le RAG](#toc1_2_)    
  - [2) Installation et configuration](#toc1_3_)    
  - [3) Construire un RAG en quelques lignes](#toc1_4_)    
  - [4) using OpenAI model for both LLM and embeddings](#toc1_5_)    
  - [5) Configuration using Ollama model for both LLM and embeddings](#toc1_6_)    
  - [6) Ollama Embeddings](#toc1_7_)    

<!-- vscode-jupyter-toc-config
	numbering=false
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

---
# <a id='toc1_'></a>[**LlamaIndex : créer son RAG en quelques lignes de code**](#toc0_)
---

## <a id='toc1_1_'></a>[Introduction](#toc0_)

LlamaIndex est un **framework Python open source spécialisé dans la création de systèmes RAG et d'applications basées sur les LLM.**

*Sa promesse* : **permettre de construire un pipeline RAG fonctionnel en quelques lignes de code, tout en offrant la flexibilité nécessaire pour les cas d'usage avancés.**

Là où d'autres frameworks adoptent une approche généraliste, LlamaIndex a fait le choix de l'excellence sur un périmètre précis : **Connecter vos Données à des Modèles de Langage.**

## <a id='toc1_2_'></a>[1) LlamaIndex : un framework pensé pour le RAG](#toc0_)

LlamaIndex est né d'un constat simple : 
- *les LLM sont puissants mais limités par leur connaissance figée au moment de leur entraînement.*
Pour les rendre utiles sur des données propriétaires - documentation interne, bases de connaissances, corpus métier - il faut les augmenter avec un système de récupération.
==> **C'est précisément ce que LlamaIndex facilite.**

Le framework s'articule autour de **cinq étapes fondamentales** qui correspondent au cycle de vie d'un système RAG :
- **Loading :** ingestion des données depuis diverses sources (fichiers, bases de données, APIs),
- **Indexing :** structuration des données pour une recherche efficace (chunking, embeddings, stockage),
- **Storing :** persistance des index pour éviter de recalculer à chaque utilisation,
- **Querying :** récupération des informations pertinentes et génération de réponses,
- **Evaluation :** mesure de la qualité du système pour l'améliorer itérativement.

## <a id='toc1_3_'></a>[2) Installation et configuration](#toc0_)

Pour utiliser LlamaIndex, il faut d'abord l'installer ainsi que ses dépendances :

In [ ]:
!pip install llama-index ipykernel llama-index-llms-ollama openai ollama

# or alternatively, using requirements.txt
!pip install --upgrade -r requirements.txt

This will install the following collected packages:
- **llama-index**: main LlamaIndex package
- **llama-index-instrumentation**: instrumentation for LlamaIndex
- **llama-index-workflows**: workflow management for LlamaIndex
- **llama-index-core**: core functionality for LlamaIndex
- **llama-index-llms-openai**: OpenAI LLM integration for LlamaIndex
- **llama-index-embeddings-openai**: OpenAI embeddings integration for LlamaIndex

Another useful packages include:
- **openai**: OpenAI API client
- **ollama**: local LLM model management and inference
- **tiktoken**: tokenization library for OpenAI models
- **nltk**: natural language toolkit for Python
- **requests**: module for making HTTP requests
- **pydantic**: data validation and settings management using Python type annotations
- **joblib**: lightweight pipelining for Python
- **jinja2**: templating engine for Python
- **httpcore**: HTTP client core for Python
- **griffe**: Python code introspection library
- **dataclasses-json**: JSON serialization for dataclasses
- **aiohttp**: asynchronous HTTP client/server framework
- **banks**: banking-related utilities

## <a id='toc1_4_'></a>[3) Construire un RAG en quelques lignes](#toc0_)

La promesse de simplicité de LlamaIndex se vérifie dans la pratique. Voici comment créer un système RAG fonctionnel à partir d'un dossier de documents :

In [ ]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
# VectorStoreIndex: permet de créer un index basé sur un magasin de vecteurs pour la recherche de similarité.
# SimpleDirectoryReader: permet de charger des documents depuis un dossier.

# Cette section montre comment créer un index de recherche basé sur des vecteurs à partir de documents stockés dans un dossier:

# Charger les documents depuis un dossier
documents = SimpleDirectoryReader("./data").load_data()

# Créer l'index (chunking + embeddings + stockage automatiques)
index = VectorStoreIndex.from_documents(documents)

# Interroger le système
query_engine = index.as_query_engine()
response = query_engine.query("Quelle est la politique de remboursement ?")

print(response)

En cinq lignes de code, vous disposez d'un RAG opérationnel:
- LlamaIndex gère automatiquement le découpage des documents en chunks, 
- la génération des embeddings via l'API OpenAI (par défaut) ou bien utiliser d'autres modèles d'embeddings (HuggingFaceEmbedding, OpenAIEmbedding, OllamaEmbedding, etc.), 
- le stockage en mémoire, 
- la recherche sémantique,
- et la génération de réponse.

Cette concision n'est pas qu'un argument marketing : **elle permet de prototyper rapidement et de valider une approche avant d'investir dans l'optimisation.**

Pour des besoins plus spécifiques, chaque composant peut être configuré individuellement :

## <a id='toc1_5_'></a>[4) using OpenAI model for both LLM and embeddings](#toc0_)

In [ ]:
# Configuration using OpenAI model for both LLM and embeddings:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Settings
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai import OpenAI
from llama_index.core.node_parser import SentenceSplitter

# Configuration globale
Settings.llm = OpenAI(model="gpt-4o", temperature=0.1)
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")

# Chunking personnalisé
node_parser = SentenceSplitter(chunk_size=512, chunk_overlap=50)

# Construction de l'index avec paramètres personnalisés
documents = SimpleDirectoryReader("./data").load_data()
nodes = node_parser.get_nodes_from_documents(documents)
index = VectorStoreIndex(nodes)

# Query engine avec paramètres de récupération
query_engine = index.as_query_engine(
    similarity_top_k=5,
    response_mode="compact"
)

Cette flexibilité progressive - simple par défaut, configurable à souhait - caractérise l'expérience développeur avec LlamaIndex. 

Vous pouvez également intégrer des modèles locaux via [Ollama](https://blent.ai/blog/a/ollama-llm) pour un RAG entièrement privé, ou connecter des *bases vectorielles* comme **Pinecone**, **Weaviate** ou **Qdrant** pour la production.

## <a id='toc1_6_'></a>[5) Configuration using Ollama model for both LLM and embeddings](#toc0_)

**Setup:**
First, follow the [readme](https://github.com/jmorganca/ollama) to set up and run a local Ollama instance.

When the Ollama app is running on your local machine:

- All of your local models are automatically served on localhost:11434
- Select your model when setting llm = Ollama(…, model=”:”)
- Increase defaullt timeout (30 seconds) if needed setting Ollama(…, request_timeout=300.0)
- If you set llm = Ollama(…, model=“<model family”) without a version it will simply look for latest
- By default, the maximum context window for your model is used. You can manually set the context_window to limit memory usage.

If you’re opening this Notebook on colab, you will probably need to install LlamaIndex 🦙.

In [ ]:
%pip install llama-index-llms-ollama ollama

In [ ]:
from llama_index.llms.ollama import Ollama

# Configuration for Ollama LLM model installed locally: ministral-3:latest, starcoder2:7b, phi4-mini:latest, qwen3.5:4b, llama3.2:latest, gemma4:31b-cloud, gemma4:latest, nomic-embed-text:latest
OLLAMA_LLM_MODEL = "llama3.2:latest"

llm = Ollama(
    model=OLLAMA_LLM_MODEL,
    request_timeout=120.0,
    # Manually set the context window to limit memory usage
    context_window=8000,
)

resp = llm.complete("Who is Paul Graham?")

print(resp)

Setup: First, follow the [readme](https://github.com/jmorganca/ollama) to set up and run a local Ollama instance.

**Gemma**: a family of lightweight, state-of-the-art open models built by Google DeepMind. Available in 2b and 7b parameter sizes

**Ollama**: Support both 2b and 7b models

**Note**: please install ollama>=0.1.26 You can download pre-release version here Ollama

When the Ollama app is running on your local machine:

- **All of your local models are automatically served on localhost:11434**
- **Select your model when setting llm = Ollama(…, model=”:”)**
- **Increase defaullt timeout (30 seconds) if needed setting Ollama(…, request_timeout=300.0)**
- **If you set llm = Ollama(…, model=“<model family”) without a version it will simply look for latest**

If you’re opening this Notebook on colab, you will probably need to install LlamaIndex 🦙.

In [ ]:
from llama_index.llms.ollama import Ollama

# gemma_2b = Ollama(model="gemma:2b", request_timeout=30.0)
gemma_2b = Ollama(model="gemma4:31b-cloud ", request_timeout=30.0)

resp = gemma_2b.complete("Who is Paul Graham?")
print(resp)

In [ ]:
# gemma_7b = Ollama(model="gemma:7b", request_timeout=30.0)
gemma_7b = Ollama(model="gemma4:latest", request_timeout=30.0*6)

resp = gemma_7b.complete("Who is Paul Graham?")
print(resp)

## <a id='toc1_7_'></a>[6) Ollama Embeddings](#toc0_)

If you’re opening this Notebook on colab, you will probably need to install LlamaIndex 🦙.

In [ ]:
%pip install llama-index-embeddings-ollama

**EmbeddingGemma** (et sa version évoluée **EmbeddingGemma 2**) est un modèle d'embedding développé par Google que vous pouvez exécuter en local via Ollama. 
Il transforme du texte, des images, de l'audio ou de la vidéo en vecteurs numériques pour la recherche sémantique et le RAG (génération augmentée par récupération).

Variantes disponibles sur Ollama:
- **EmbeddingGemma 1 (300M)** : Modèle centré sur le texte uniquement (environ 622 Mo, fenêtre de 2 000 tokens).
- **EmbeddingGemma 2** : Modèle multimodal basé sur Gemma 4 proposé en plusieurs tailles selon vos besoins :
	- embeddinggemma-2:270m (~304–378 Mo, texte seul)
	- embeddinggemma-2:440m (~671–714 Mo, texte et image)
	- embeddinggemma-2:570m (~918–990 Mo, texte)
	- embeddinggemma-2:740m (étiqueté :latest, ~1,3 Go, texte et image)

Commandes principales:
- Télécharger le modèle :

In [ ]:
!ollama pull embeddinggemma-2

- Utilisation en Python :

In [ ]:
import ollama

response = ollama.embed(
    model='embeddinggemma-2',
    input='Pourquoi le ciel est bleu ?',
)
print(response['embeddings'])

Sample example using Ollama embeddings:

In [ ]:
from llama_index.embeddings.ollama import OllamaEmbedding

# Configuration for Ollama LLM model installed locally: ministral-3:latest, starcoder2:7b, phi4-mini:latest, qwen3.5:4b, llama3.2:latest, gemma4:31b-cloud, gemma4:latest, nomic-embed-text:latest
# Ollama embedding model installed locally:embeddinggemma-2, embeddinggemma-2:latest, nomic-embed-text:latest, embeddinggemma:300m
# Use !ollama pull <model_name> to download the model locally if not already installed
OLLAMA_LLM_MODEL = "embeddinggemma-2"

ollama_embedding = OllamaEmbedding(
    model_name=OLLAMA_LLM_MODEL,  # ou autre modèle comme "embeddinggemma"
    base_url="http://localhost:11434",
    # Can optionally pass additional kwargs to ollama
    # ollama_additional_kwargs={"mirostat": 0},
)

# where:
# model_name: The name of the Ollama embedding model to use, e.g., "embeddinggemma"
# base_url: The base URL of the Ollama server, e.g., "http://localhost:11434"
# ollama_additional_kwargs: Optional additional keyword arguments to pass to Ollama, e.g., {"mirostat": 0}

In [ ]:
help(ollama_embedding.get_text_embedding_batch)

You can generate embeddings using one of several methods:
- get_text_embedding_batch
- get_text_embedding
- get_query_embedding

As well as async versions:
- aget_text_embedding_batch
- aget_text_embedding
- aget_query_embedding

In [ ]:
embeddings = ollama_embedding.get_text_embedding_batch(
    ["This is a passage!", "This is another passage"], show_progress=True
)

print(f"Got vectors of length {len(embeddings[0])}")
print(embeddings[0][:10])

In [ ]:
embedding = ollama_embedding.get_text_embedding(
    "This is a piece of text!",
)
print(f"Got vectors of length {len(embedding)}")
print(embedding[:10])

In [ ]:
embedding = ollama_embedding.get_query_embedding(
    "This is a query!",
)
print(f"Got vectors of length {len(embedding)}")
print(embedding[:10])

Le module llama_index.embeddings.ollama permet d'utiliser des modèles d'embedding hébergés localement via Ollama au sein du framework LlamaIndex. Il offre des fonctions pour obtenir des embeddings de texte, que ce soit pour une seule requête ou pour un lot de passages, facilitant ainsi l'intégration des modèles d'embedding locaux dans les pipelines de traitement de texte et de recherche d'information.

In [ ]:
from llama_index.embeddings.ollama import OllamaEmbedding

# Ollama embedding model installed locally:embeddinggemma-2, embeddinggemma-2:latest, nomic-embed-text:latest, embeddinggemma:300m
# Use !ollama pull <model_name> to download the model locally if not already installed
OLLAMA_LLM_MODEL = "embeddinggemma-2:latest"

# Initialisation du modèle d'embedding
embed_model = OllamaEmbedding(
    model_name=OLLAMA_LLM_MODEL,  # ou autre modèle comme "embeddinggemma"
    base_url="http://localhost:11434",  # URL par défaut d'Ollama
)

# Générer un embedding pour un texte
text_embedding = embed_model.get_text_embedding("Hello, world!")
print(f"Embedding dimension : {len(text_embedding)}")

# Générer un embedding pour une requête
query_embedding = embed_model.get_query_embedding("What is AI?")
print(f"Query embedding dimension : {len(query_embedding)}")

Principaux paramètres:
- **model_name** : Le nom du modèle d'embedding exécuté dans Ollama (ex: nomic-embed-text, embeddinggemma).
- **base_url** : L'adresse du serveur Ollama (par défaut http://localhost:11434).
- **embed_batch_size** : La taille des lots pour le traitement (entre 1 et 2048).
- **keep_alive** : Durée de maintien du modèle en mémoire (par défaut 5m).

Vous pouvez consulter la documentation officielle sur la page **Ollama Embeddings - LlamaParse** ou la fiche sur le PyPI **llama-index-embeddings-ollama**

reprennons l'exemple **avec** des documents de données robotiques:

In [ ]:
# Configuration using Ollama model for both LLM and embeddings:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Settings
from llama_index.embeddings.ollama import OllamaEmbedding
from llama_index.llms.ollama import Ollama
from llama_index.core.node_parser import SentenceSplitter

# Ollama embedding model installed locally:embeddinggemma-2, embeddinggemma-2:latest, nomic-embed-text:latest, embeddinggemma:300m
# Use !ollama pull <model_name> to download the model locally if not already installed
OLLAMA_LLM_MODEL = "embeddinggemma-2:latest"

# Configuration globale
Settings.llm = Ollama(model=OLLAMA_LLM_MODEL, temperature=0.1)
Settings.embed_model = OllamaEmbedding(model_name=OLLAMA_LLM_MODEL)

# Chunking personnalisé
node_parser = SentenceSplitter(chunk_size=512, chunk_overlap=50)

# Construction de l'index avec paramètres personnalisés
documents = SimpleDirectoryReader("./data").load_data()
nodes = node_parser.get_nodes_from_documents(documents, show_progress=True)
index = VectorStoreIndex(nodes, show_progress=True)

# Query engine avec paramètres de récupération
query_engine = index.as_query_engine(
    similarity_top_k=5,
    response_mode="compact"
)

# Exécution d'une requête sur l'index
query = "Quels sont les derniers développements en robotique ?"
response = query_engine.query(query)
print(response)

question 2:

In [ ]:
query2 = "Quels sont les principes de base en robotique ?"
response2 = query_engine.query(query2)
print(response2)

question 3:

In [ ]:
query3 = "Quels sont les éléments de base en robotique ?"
response3 = query_engine.query(query3)
print(response3)

question 4:

In [ ]:
query4 = "Quels sont les défis actuels en robotique ?"
response4 = query_engine.query(query4)
print(response4)

question 5:

In [ ]:
query5 = "Comment construire les matrices en robotique ?"
response5 = query_engine.query(query5)
print(response5)